In [128]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\sathwika\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [129]:
import numpy as np
import pandas as pd
import plotly.express as pxp
import plotly.io as pio
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import webbrowser
import os


In [130]:
import os

print("Current Folder:", os.getcwd())
print(os.path.exists("../datasets/googleplaystore.csv"))

Current Folder: c:\Users\sathwika\OneDrive\Documents\Desktop\elevance_internship_training\task1
True


In [131]:
import pandas as pd
apps_df = pd.read_csv("../datasets/googleplaystore.csv")
reviews_df = pd.read_csv("../datasets/googleplaystore_user_reviews.csv")

In [132]:
reviews_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [133]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [134]:
merged_df = pd.merge(apps_df, reviews_df, on="App", how="inner")

In [135]:
merged_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,A kid's excessive ads. The types ads allowed a...,Negative,-0.250,1.000000
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,It bad >:(,Negative,-0.725,0.833333
2,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,like,Neutral,0.000,0.000000
3,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,NaN,NaN,NaN,NaN
4,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,I love colors inspyering,Positive,0.500,0.600000


In [136]:
merged_df.shape

(122662, 17)

In [137]:
merged_df.columns

Index(['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type',
       'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver',
       'Android Ver', 'Translated_Review', 'Sentiment', 'Sentiment_Polarity',
       'Sentiment_Subjectivity'],
      dtype='object')

In [138]:
merged_df.to_csv("merged_googleplaystore_data.csv", index=False)

In [139]:
print(merged_df["App"].value_counts().head(10))

App
CBS Sports App - Scores, News, Stats & Watch Live    2560
8 Ball Pool                                          2100
Bowmasters                                           1920
Helix Jump                                           1800
Candy Crush Saga                                     1680
Duolingo: Learn Languages Free                       1680
ESPN                                                 1680
Angry Birds Classic                                  1600
Bubble Shooter                                       1560
Calorie Counter - MyFitnessPal                       1300
Name: count, dtype: int64


In [140]:
avg_subjectivity = (
    reviews_df.groupby("App")["Sentiment_Subjectivity"]
    .mean()
    .reset_index()
)

In [141]:
merged_df = pd.merge(
    apps_df,
    avg_subjectivity,
    on="App",
    how="inner"
)

In [142]:
apps_df.shape


(10841, 13)

In [143]:
merged_df.shape

(1532, 14)

In [144]:
merged_df["Reviews"] = pd.to_numeric(merged_df["Reviews"], errors="coerce")

In [145]:
merged_df["Rating"] = pd.to_numeric(merged_df["Rating"], errors="coerce")

In [146]:
merged_df["Installs"] = (
    merged_df["Installs"]
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
    .astype(int)
)

In [147]:
merged_df["Size"] = (
    merged_df["Size"]
    .str.replace("M", "", regex=False)
)

In [148]:
merged_df["Size"] = pd.to_numeric(merged_df["Size"], errors="coerce")

In [149]:
print(merged_df[["Rating","Reviews","Installs","Size"]].isnull().sum())

Rating        1
Reviews       0
Installs      0
Size        529
dtype: int64


In [150]:
merged_df = merged_df.dropna(subset=[
    "Rating",
    "Size",
    "Sentiment_Subjectivity"
])

In [151]:
print(merged_df.isnull().sum())
print(merged_df.shape)

App                       0
Category                  0
Rating                    0
Reviews                   0
Size                      0
Installs                  0
Type                      0
Price                     0
Content Rating            0
Genres                    0
Last Updated              0
Current Ver               0
Android Ver               0
Sentiment_Subjectivity    0
dtype: int64
(805, 14)


In [152]:
translations = {
    "BEAUTY": "सौंदर्य",
    "BUSINESS": "வணிகம்",
    "DATING": "Dating"
}

In [153]:
merged_df["Category_Display"] = merged_df["Category"].replace(translations)

In [154]:
print(merged_df[["Category", "Category_Display"]].drop_duplicates())

                 Category     Category_Display
0          ART_AND_DESIGN       ART_AND_DESIGN
12      AUTO_AND_VEHICLES    AUTO_AND_VEHICLES
21                 BEAUTY              सौंदर्य
32    BOOKS_AND_REFERENCE  BOOKS_AND_REFERENCE
54               BUSINESS               வணிகம்
94                 COMICS               COMICS
101         COMMUNICATION        COMMUNICATION
148                DATING               Dating
227             EDUCATION            EDUCATION
269         ENTERTAINMENT        ENTERTAINMENT
319                EVENTS               EVENTS
326               FINANCE              FINANCE
376        FOOD_AND_DRINK       FOOD_AND_DRINK
399    HEALTH_AND_FITNESS   HEALTH_AND_FITNESS
467        HOUSE_AND_HOME       HOUSE_AND_HOME
492    LIBRARIES_AND_DEMO   LIBRARIES_AND_DEMO
503             LIFESTYLE            LIFESTYLE
534                  GAME                 GAME
701                FAMILY               FAMILY
779               MEDICAL              MEDICAL
838          

In [155]:
required_categories = [
    "GAME",
    "BEAUTY",
    "BUSINESS",
    "COMICS",
    "COMMUNICATION",
    "DATING",
    "ENTERTAINMENT",
    "SOCIAL",
    "EVENTS"
]

filtered_df = merged_df[
    (merged_df["Rating"] > 3.5) &
    (merged_df["Reviews"] > 500) &
    (merged_df["Installs"] > 50000) &
    (merged_df["Sentiment_Subjectivity"] > 0.5) &
    (merged_df["Category"].isin(required_categories)) &
    (~merged_df["App"].str.contains("S", case=False, na=False))
]

In [156]:
print(filtered_df.shape)
print(filtered_df.head())

(42, 15)
                                                   App       Category  Rating  \
55                                       Google Primer       BUSINESS     4.4   
58                                        Call Blocker       BUSINESS     4.6   
134                                       Call Blocker  COMMUNICATION     4.1   
135  CallApp: Caller ID, Blocker & Phone Call Recorder  COMMUNICATION     4.4   
148          Hily: Dating, Chat, Match, Meet & Hook up         DATING     4.1   

     Reviews  Size  Installs  Type Price Content Rating         Genres  \
55     62272  18.0  10000000  Free     0       Everyone       Business   
58    188841   3.2   5000000  Free     0       Everyone       Business   
134    17529  10.0   1000000  Free     0       Everyone  Communication   
135   483565  20.0  10000000  Free     0       Everyone  Communication   
148     2556  56.0    100000  Free     0     Mature 17+         Dating   

       Last Updated Current Ver   Android Ver  Sentiment_Su

In [157]:
print(filtered_df["Category_Display"].unique())

['வணிகம்' 'COMMUNICATION' 'Dating' 'ENTERTAINMENT' 'GAME' 'SOCIAL']


In [158]:
import plotly.express as px


In [159]:
color_map = {
    "GAME": "pink",
    "सौंदर्य": "orange",
    "வணிகம்": "green",
    "COMICS": "purple",
    "COMMUNICATION": "blue",
    "Dating": "red",
    "ENTERTAINMENT": "gold",
    "SOCIAL": "cyan",
    "EVENTS": "brown"
}

fig = px.scatter(
    filtered_df,
    x="Size",
    y="Rating",
    size="Installs",
    color="Category_Display",
    hover_data=[
    "App",
    "Installs",
    "Reviews",
    "Category_Display"
],
    color_discrete_map=color_map,
    title="Google Play Store Analytics\n App Size vs Average Rating"
)
fig.update_layout(
    xaxis_title="App Size (MB)",
    yaxis_title="Average Rating"
)
fig.update_traces(
    marker=dict(
        line=dict(width=1,color="black")
    )
)
fig.update_layout(
    title={
        "text": "Google Play Store Analytics<br><sup>App Size vs Average Rating</sup>",
        "x": 0.5,
        "xanchor": "center"
    },
    xaxis_title="App Size (MB)",
    yaxis_title="Average Rating",
    width=1200,
    height=650,
    template="plotly_white"
)

fig.update_layout(
    width=1200,
    height=650
)

In [160]:
from datetime import datetime, time
import pytz

ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist).time()

if time(17, 0) <= current_time <= time(19, 0):
    fig.show()
else:
    print("Bubble Chart is available only between 5 PM and 7 PM IST.")